In [17]:
import pandas as pd

In [18]:
stats= pd.read_csv(r"C:\Users\Owner\Documents\NSS\PYTHON\data\season_stats - season_stats.csv")

In [ ]:
stats.head(5)

In [19]:
stats_clean= pd.read_csv(
    r"C:\Users\Owner\Documents\NSS\PYTHON\data\season_stats - season_stats.csv",
    header=None,
    names=["team_id", "week", "name", "opp_name", "tm_score", "opp_score", "rush_yds", "pass_yds"]
)

stats_clean = stats_clean[~stats_clean["week"].astype(str).str.lower().eq("week")].copy()
stats_clean["rush_yds"] = pd.to_numeric(stats_clean["rush_yds"], errors="coerce")

In [ ]:
stats_clean.head(5)

In [20]:
stats2= stats_clean.copy()

In [ ]:
stats2.head(5)

In [21]:
stats3= stats_clean.copy()

In [22]:
stats4= stats_clean.copy()

Claude understanding and interpreting it

In [23]:
import pandas as pd

null_counts = stats_clean.isna().sum()
rows_with_nulls = stats_clean[stats_clean.isna().any(axis=1)].copy()

print('null_counts')
print(null_counts)
print('\nrows_with_any_null', len(rows_with_nulls))
print('\nrows_with_nulls_by_name')
print(rows_with_nulls['name'].value_counts().head(10).to_string())
print('\nrows_with_nulls_by_week')
print(rows_with_nulls['week'].value_counts().head(10).to_string())
print('\nrows_with_nulls_sample')
print(rows_with_nulls.head(10).to_string(index=False))

null_counts
team_id       0
week          0
name          0
opp_name      0
tm_score      0
opp_score     0
rush_yds     16
pass_yds      0
dtype: int64

rows_with_any_null 16

rows_with_nulls_by_name
name
49ers       1
Bears       1
Broncos     1
Chiefs      1
Colts       1
Cowboys     1
Dolphins    1
Eagles      1
Falcons     1
Giants      1

rows_with_nulls_by_week
week
12    16

rows_with_nulls_sample
 team_id week     name      opp_name tm_score opp_score  rush_yds pass_yds
   117.0   12    49ers      Seahawks       31        13       NaN      209
   104.0   12    Bears       Vikings       12        10       NaN      217
   111.0   12  Broncos        Browns       29        12       NaN      134
   106.0   12   Chiefs       Raiders       31        17       NaN      298
   113.0   12    Colts    Buccaneers       27        20       NaN      251
   119.0   12  Cowboys Football Team       45        10       NaN      331
   116.0   12 Dolphins          Jets       34        13       NaN 

In [ ]:
print('columns:', list(stats_clean.columns))
print('\nhead:')
print(stats.head(5).to_string())
print('\nnull_counts:')
print(stats_clean.isna().sum().to_string())
rows_with_nulls = stats_clean[stats_clean.isna().any(axis=1)].copy()
print('\nrows_with_any_null', len(rows_with_nulls))
print(rows_with_nulls.head(20).to_string())

Stats2: replace nulls with avg of rush_yds

In [24]:
# capture which rows are NaN BEFORE filling
na_mask = stats2["rush_yds"].isna()

rush_mean = stats2["rush_yds"].mean()
stats2["rush_yds"] = stats2["rush_yds"].fillna(rush_mean).astype("int64")

# now pull those same rows — they'll have the new filled value
filled_rows2 = stats2[na_mask].copy()

print('rows that were filled:', filled_rows2.shape[0])
print(filled_rows2.head(5).to_string(index=False))

rows that were filled: 16
 team_id week    name   opp_name tm_score opp_score  rush_yds pass_yds
   117.0   12   49ers   Seahawks       31        13       126      209
   104.0   12   Bears    Vikings       12        10       126      217
   111.0   12 Broncos     Browns       29        12       126      134
   106.0   12  Chiefs    Raiders       31        17       126      298
   113.0   12   Colts Buccaneers       27        20       126      251


In [25]:
print(filled_rows2)

     team_id week      name       opp_name tm_score opp_score  rush_yds  \
4      117.0   12     49ers       Seahawks       31        13       126   
15     104.0   12     Bears        Vikings       12        10       126   
41     111.0   12   Broncos         Browns       29        12       126   
79     106.0   12    Chiefs        Raiders       31        17       126   
90     113.0   12     Colts     Buccaneers       27        20       126   
100    119.0   12   Cowboys  Football Team       45        10       126   
111    116.0   12  Dolphins           Jets       34        13       126   
120    107.0   12    Eagles          Bills       37        34       126   
132    115.0   12   Falcons         Saints       24        15       126   
143    109.0   12    Giants       Patriots       10         7       126   
148    110.0   12   Jaguars         Texans       24        21       126   
180    118.0   12   Packers          Lions       29        22       126   
204    112.0   12      Ra

In [26]:
#check to see if rush_yds is integer type
stats2['rush_yds'].dtype

dtype('int64')

In [ ]:
stats2.head(10)

Pros of this method
Very simple and quick to implement
Keeps the column numeric and consistent
Works well when missing values are a small fraction of the data
Preserves the overall column distribution better than dropping rows
Cons of this method
It can distort the data because the same average value is inserted into many rows
It may hide true differences between rows
If the missing values are not random, mean imputation can bias your downstream analysis
It is less accurate than more advanced methods like group-based imputation or model-based prediction

Stats3: cleanup with team avarage

In [27]:

stats3 = stats3[~stats3["week"].astype(str).str.lower().eq("week")].copy()
stats3["rush_yds"] = pd.to_numeric(stats3["rush_yds"], errors="coerce")

na_mask = stats3["rush_yds"].isna()  # capture before filling

team_rush_mean = stats3.groupby("name")["rush_yds"].transform("mean")
stats3["rush_yds"] = stats3["rush_yds"].fillna(team_rush_mean).astype("int64")

filled_rows3 = stats3[na_mask].copy()

print('rows filled:', filled_rows3.shape[0])
print(filled_rows3.head(5).to_string(index=False))

rows filled: 16
 team_id week    name   opp_name tm_score opp_score  rush_yds pass_yds
   117.0   12   49ers   Seahawks       31        13       153      209
   104.0   12   Bears    Vikings       12        10       178      217
   111.0   12 Broncos     Browns       29        12       110      134
   106.0   12  Chiefs    Raiders       31        17       108      298
   113.0   12   Colts Buccaneers       27        20       120      251


In [28]:
#check to see if there are any nulls in the rush_yds column 
stats3['rush_yds'].isnull().sum()
#check to see data type of rush_yds column
stats3['rush_yds'].dtype


dtype('int64')

In [30]:
stats3.head(3)

,team_id,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
0,28.0,17,49ers,Football Team,27,10,184,230
1,68.0,15,49ers,Cardinals,45,29,144,262
2,78.0,14,49ers,Seahawks,28,16,173,368


Pros
Better than a single overall mean for team-based sports data
Keeps the dataset complete and numeric
Preserves more realistic team-level differences
Easy to understand and explain
Cons
Still an estimate, not a true observed value
Can make missing rows appear more “typical” than they really were
If a team has very few games, its average can be unstable
If the missingness is not random, this method can still introduce bias

Stats 4: Replace nulls with predicted vlaues using linear regression
Linear regression- ML method drwas a straight line pattern b/w input vals(name,tm_score...) and target vals(rush_yds)

Pros
Uses multiple features, so it can make better estimates than a single mean
More realistic than filling every missing value with one number
Good as a simple baseline predictive method
Keeps the dataset complete for downstream modeling
Cons
It is still only a prediction, not a true observation
Linear regression is a simple model and may miss nonlinear relationships
Team name as a category is helpful, but the model may still be limited by data size
If the missingness is structured, the model could be biased.

In [33]:
from sklearn.linear_model import LinearRegression

for col in ["team_id", "week", "tm_score", "opp_score", "rush_yds", "pass_yds"]:
    stats4[col] = pd.to_numeric(stats4[col], errors="coerce")

train = stats4.dropna(subset=["rush_yds"]).copy()
predict_rows4 = stats4[stats4["rush_yds"].isna()].copy()  # the "before" rows, same role as filled_rows earlier

if len(predict_rows4) == 0:
    stats4 = train.copy()
else:
    features = ["name", "tm_score", "opp_score", "pass_yds"]
    X_train = pd.get_dummies(train[features], columns=["name"])
    X_pred = pd.get_dummies(predict_rows4[features], columns=["name"])
    X_train, X_pred = X_train.align(X_pred, join="outer", axis=1, fill_value=0)

    model = LinearRegression().fit(X_train, train["rush_yds"])
    predict_rows4["rush_yds"] = model.predict(X_pred).round().astype("int64")

    stats4 = pd.concat([train, predict_rows4]).sort_index()
    stats4["rush_yds"] = stats4["rush_yds"].astype("int64")

print('stats4 shape:', stats4.shape)
print('rows predicted:', len(predict_rows4))
print(predict_rows4.head(5)[["name", "tm_score", "opp_score", "pass_yds", "rush_yds"]].to_string(index=False))

stats4 shape: (272, 8)
rows predicted: 16
   name  tm_score  opp_score  pass_yds  rush_yds
  49ers        31         13       209       165
  Bears        12         10       217       138
Broncos        29         12       134       142
 Chiefs        31         17       298       116
  Colts        27         20       251       118


In [36]:
#check to see if there are any nulls in the rush_yds column
stats4['rush_yds'].isnull().sum()

0

In [38]:
predict_rows4

,team_id,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
4,117.0,12,49ers,Seahawks,31,13,165,209
15,104.0,12,Bears,Vikings,12,10,138,217
41,111.0,12,Broncos,Browns,29,12,142,134
79,106.0,12,Chiefs,Raiders,31,17,116,298
90,113.0,12,Colts,Buccaneers,27,20,118,251
100,119.0,12,Cowboys,Football Team,45,10,132,331
111,116.0,12,Dolphins,Jets,34,13,157,243
120,107.0,12,Eagles,Bills,37,34,165,200
132,115.0,12,Falcons,Saints,24,15,161,168
143,109.0,12,Giants,Patriots,10,7,114,191


Actual data:

In [34]:
actual= pd.read_csv(r"C:\Users\Owner\Documents\NSS\PYTHON\data\actual.csv")

In [39]:
actual.head(5)

,week,name,actual_rush_yds
0,18,Cowboys,131
1,18,Rams,109
2,18,Chiefs,123
3,18,Raiders,129
4,18,Titans,175


In [45]:
actual['week'].dtype

dtype('int64')

Mean Absolute Error w/ ChatGPT

In [46]:
#object to int
filled_rows2['week'] = filled_rows2['week'].astype(int)
#stats2 mean abs error
compare2 = filled_rows2.merge(
    actual[["week", "name", "actual_rush_yds"]],
    on=["week", "name"],
    how="inner"
)

In [47]:
filled_rows3['week'] = filled_rows3['week'].astype(int)

compare3 = filled_rows3.merge(
    actual[["week", "name", "actual_rush_yds"]],
    on=["week", "name"],
    how="inner"
)

In [50]:
predict_rows4['week'] = predict_rows4['week'].astype(int)


compare4 = predict_rows4.merge(
    actual[["week", "name", "actual_rush_yds"]],
    on=["week", "name"],
    how="inner"
)

In [51]:
#calc
from sklearn.metrics import mean_absolute_error

mae_avg = mean_absolute_error(
    compare2["actual_rush_yds"],
    compare2["rush_yds"]
)

mae_team = mean_absolute_error(
    compare3["actual_rush_yds"],
    compare3["rush_yds"]
)

mae_regression = mean_absolute_error(
    compare4["actual_rush_yds"],
    compare4["rush_yds"]
)

print(f"Overall Average MAE: {mae_avg:.2f}")
print(f"Team Average MAE: {mae_team:.2f}")
print(f"Regression MAE: {mae_regression:.2f}")

Overall Average MAE: 49.00
Team Average MAE: 46.50
Regression MAE: 35.81


Claude Mean squared Error

In [53]:
from sklearn.metrics import mean_squared_error

def rush_yds_mse(pred_df, label):
    merged = actual.merge(pred_df[["week", "name", "rush_yds"]], on=["week", "name"], how="inner")
    mse = mean_squared_error(merged["actual_rush_yds"], merged["rush_yds"])
    print(f"{label}: MSE = {mse:.2f}  (n={len(merged)})")
    return mse

results = {
    "stats2 (overall mean fill)":  rush_yds_mse(filled_rows2, "stats2 (overall mean fill)"),
    "stats3 (team mean fill)":     rush_yds_mse(filled_rows3, "stats3 (team mean fill)"),
    "stats4 (regression predict)": rush_yds_mse(predict_rows4, "stats4 (regression predict)"),
}

stats2 (overall mean fill): MSE = 3082.62  (n=16)
stats3 (team mean fill): MSE = 2706.50  (n=16)
stats4 (regression predict): MSE = 1705.81  (n=16)
